# unimovie — quickstart

Demultiplexing for video containers: what is inside a file, without decoding any of it.

This notebook writes the file it then inspects, so it runs anywhere the wheel is installed — no fixture from the source tree, which is also how CI executes it against the published wheel.

In [1]:
import os
import tempfile

import unimovie

os.chdir(tempfile.mkdtemp())
spec = [
    {'kind': 'video', 'codec': 'avc1', 'timescale': 1000,
     'width': 64, 'height': 48},
    {'kind': 'audio', 'codec': 'mp4a', 'timescale': 44100,
     'channels': 2, 'sample_rate': 44100},
]
# The samples are opaque bytes: nothing here encodes or decodes.
with unimovie.open_writer('demo.mp4', spec) as writer:
    for index in range(3):
        writer.write(0, b'video-sample-%d' % index, 40,
                     keyframe=(index == 0))
    for index in range(2):
        writer.write(1, b'audio-sample-%d' % index, 1024)

FIXTURE = 'demo.mp4'
unimovie.version()

'0.1.0'

`probe` gives the shape in five values: how many tracks, which is the first video and audio one — `-1` when there is none, because a file without sound is an ordinary file — the playing time, and the container. An ISO base media file answers with the major brand it claims rather than a guess at its extension: `mp42` here, because that is what the writer stamped into it.

In [2]:
unimovie.probe(FIXTURE)

(2, 0, 1, 0.12, 'mp42')

`tracks` lists every track. `codec` is the container's own four-character code rather than a friendlier name: that is what a decoder backend is registered under, and `avc1` and `avc3` differ in where their parameter sets live.

In [3]:
unimovie.tracks(FIXTURE)

[{'kind': 'video',
  'codec': 'avc1',
  'width': 64,
  'height': 48,
  'rotation': 0,
  'sample_count': 3,
  'keyframe_count': 1},
 {'kind': 'audio',
  'codec': 'mp4a',
  'width': 0,
  'height': 0,
  'rotation': 0,
  'sample_count': 2,
  'keyframe_count': 2}]

`rotation`, zero above because nothing asked for a transform, counts clockwise degrees. `ffprobe` reports the same transformation matrix counting anticlockwise, so a file it calls `rotation=90` reads here as `270`. Both describe the same matrix; the sign is the trap when migrating off `ffprobe`.

Nothing above decoded a frame. Turning a coded sample into pixels belongs to a backend the application registers, which is what keeps a patented decoder out of this library.

See `include/UniMovie.h`, and the book for the full picture.